# felix_05 — repeat dosage, end to end on toy data

Five samples, one catalog locus, three overlapping repeat records. Small enough that
every intermediate object can be printed in full, and arranged so the numbers match
`repeat_dosage_schematic.png` exactly.

The notebook **drives the production code** — `scripts/aggregate_repeat_loci.py` and
`scripts/write_admixed_dosage_vcf.py` — and never restates their logic, so anything
printed here is what the pipeline actually computes.

| stage | what happens | object to inspect |
|---|---|---|
| Level 1 | sum signed bp change per sample × per haplotype | `LocusState.sum1` / `.sum2` |
| Level 2 | distinct cohort-wide haplotype totals become the ALT alleles | `loci.alleles.tsv` and the locus VCF record |
| Level 3 | per ancestry, **sum** the dosages and **count** the haplotypes | `site_dosages()` → `ds_rows`, `anc_rows` |
| test | FELIX step 2 reads `FORMAT DS1..DSK` with `ANC1..ANCK` | the admixed VCF |

Requires `bcftools` on `PATH` (the aggregator streams records through `bcftools query`).

In [ ]:
import gzip, io, json, os, shutil, subprocess, sys, textwrap
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo() -> Path:
    env = os.environ.get("FELIX_REPO")
    cands = [Path(env)] if env else []
    cands += [Path.cwd(), *Path.cwd().parents]
    cands += [Path.home() / "aou-lr-phase-2",
              Path("/home/jupyter/AoU_DRC_LongReads_PhaseTwo_Storage/edit")]
    for base in cands:
        if (base / "scripts" / "aggregate_repeat_loci.py").exists():
            return base.resolve()
    raise SystemExit("set FELIX_REPO to the aou-lr-phase-2 checkout")

REPO = find_repo()
sys.path.insert(0, str(REPO / "scripts"))

if shutil.which("bcftools") is None:
    raise SystemExit("bcftools not on PATH: conda install -c bioconda bcftools")

import aggregate_repeat_loci as agg
import write_admixed_dosage_vcf as wad

WORK = Path("felix_dosage_demo").resolve()
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

print("repo    ", REPO)
print("bcftools", subprocess.run(["bcftools", "--version"], capture_output=True,
                                 text=True).stdout.splitlines()[0])
print("work    ", WORK)

## Level 0 — the toy input

One catalog locus, `chr1:1000-1060`, motif `CAG`, so `period = 3`.

Three repeat records fall inside it. Each is multi-allelic so that a single record can
carry both a contraction and an expansion:

| record | POS | REF | ALT | allele 1 | allele 2 |
|---|---|---|---|---|---|
| `p1` | 1015 | `ACAG` | `A,ACAGCAGCAG` | −3 bp | +6 bp |
| `p2` | 1030 | `ACAG` | `A,ACAGCAG` | −3 bp | +3 bp |
| `p3` | 1045 | `ACAG` | `A,ACAGCAG` | −3 bp | +3 bp |

Per-haplotype genotypes are chosen to give the four interesting cases:

- **S1** one expansion on one haplotype
- **S2** an expansion assembled from *two* records on haplotype 1, a contraction on haplotype 2
- **S3** a contraction, on a haplotype whose local ancestry differs from its partner's
- **S4** +6 bp and −6 bp on two haplotypes **of the same ancestry** — the cancellation case
- **S5** reference everywhere — the control S4 will turn out to be indistinguishable from

`AN1`/`AN2` are the FLARE local-ancestry codes: `0 = AFR`, `1 = EUR`, `2 = AMR`.

In [ ]:
HEADER = """##fileformat=VCFv4.2
##contig=<ID=chr1,length=248956422>
##INFO=<ID=SVTYPE,Number=1,Type=String,Description="Type">
##INFO=<ID=SVLEN,Number=A,Type=Integer,Description="Length">
##INFO=<ID=END,Number=1,Type=Integer,Description="End">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=AN1,Number=1,Type=Integer,Description="Local ancestry, haplotype 1">
##FORMAT=<ID=AN2,Number=1,Type=Integer,Description="Local ancestry, haplotype 2">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tS1\tS2\tS3\tS4\tS5
"""

SAMPLES = ["S1", "S2", "S3", "S4", "S5"]
ANC = {"S1": (0, 0), "S2": (0, 1), "S3": (1, 2), "S4": (0, 0), "S5": (0, 0)}

#                                              S1     S2     S3     S4     S5
RECORDS = [
    ("1015", "p1", "ACAG", "A,ACAGCAGCAG", ["2|0", "0|1", "1|0", "2|0", "0|0"]),
    ("1030", "p2", "ACAG", "A,ACAGCAG",    ["0|0", "2|0", "0|0", "0|1", "0|0"]),
    ("1045", "p3", "ACAG", "A,ACAGCAG",    ["0|0", "2|0", "0|0", "0|1", "0|0"]),
]

lines = [HEADER.rstrip("\n")]
for pos, rid, ref, alt, gts in RECORDS:
    cells = [f"{g}:{ANC[s][0]}:{ANC[s][1]}" for g, s in zip(gts, SAMPLES)]
    lines.append("\t".join(["chr1", pos, rid, ref, alt, ".", "PASS", ".", "GT:AN1:AN2", *cells]))
plain = WORK / "toy.vcf"
plain.write_text("\n".join(lines) + "\n")

VCF = WORK / "toy.vcf.gz"
subprocess.run(["bcftools", "view", "-Oz", "-o", str(VCF), str(plain)], check=True)
subprocess.run(["bcftools", "index", "-t", str(VCF)], check=True)

BED = WORK / "catalog.bed"
BED.write_text("chr1\t1000\t1060\tL1\tCAG\n")

print(plain.read_text())
print("catalog:", BED.read_text().strip())

### Signed bp per allele index

`allele_bp()` is the only place a length change is derived. It returns one signed value
per allele index, with index 0 (REF) pinned at 0 — this is where the sign convention is
set, and it is never undone downstream.

In [ ]:
rows = []
for pos, rid, ref, alt, _gts in RECORDS:
    bp = agg.allele_bp(ref, alt.split(","), ".", ".", ".", int(pos))
    rows.append({"record": rid, "POS": int(pos), "REF": ref, "ALT": alt,
                 "alt_bp (by allele index)": bp.tolist()})
pd.DataFrame(rows).set_index("record")

## Level 1 — `LocusState.sum1` and `LocusState.sum2`

`Aggregator.finish(st)` is called once per locus with the fully accumulated `LocusState`.
Patching it to stash `st` before delegating gives us the real object, with no
reimplementation: `sum1` and `sum2` are separate `int64` accumulators, and
`LocusState.add()` walks `(rec.a1, sum1)` and `(rec.a2, sum2)` independently, so the two
haplotypes never mix.

In [ ]:
CAPTURED = {}
_orig_finish = agg.Aggregator.finish

def capturing_finish(self, st):
    CAPTURED[st.locus.lid] = st
    return _orig_finish(self, st)

agg.Aggregator.finish = capturing_finish
try:
    rc = agg.main([
        "--vcf", str(VCF), "--catalog-bed", str(BED), "--chrom", "chr1",
        "--with-ancestry", "--num-ancs", "3",
        "--out-alleles", str(WORK / "loci.alleles.tsv"),
        "--out-vcf",     str(WORK / "loci.vcf"),
        "--out-summary", str(WORK / "loci.summary.json"),
    ])
finally:
    agg.Aggregator.finish = _orig_finish
assert rc == 0, rc

st = CAPTURED["L1"]
print("locus      ", st.locus)
print("period     ", st.locus.period, "bp")
print("n_records  ", st.n_records)

level1 = pd.DataFrame({
    "sum1_bp":  st.sum1,      "sum2_bp":  st.sum2,
    "miss1":    st.miss1,     "miss2":    st.miss2,
    "nrec1":    st.nrec1,     "nrec2":    st.nrec2,
    "AN1":      st.an1,       "AN2":      st.an2,
}, index=SAMPLES)
level1

`nrec1`/`nrec2` count how many records contributed a non-zero change to that haplotype —
S2's haplotype 1 and S4's haplotype 2 each show 2, which is the "multiple records on one
haplotype add together" case. `miss*` is all-False here (no missing calls, no ambiguous
unphased heterozygotes), so every haplotype is called.

## Level 2 — distinct haplotype totals become the ALT alleles

`finish()` takes `np.unique` over **every called haplotype in the cohort** and writes one
allele-table row per distinct total. That table is the aggregator's own record of the step,
so read it rather than recomputing the unique values.

In [ ]:
alleles = pd.read_csv(WORK / "loci.alleles.tsv", sep="\t")
alleles

`n_hap_allele` is the per-allele haplotype count — the only place in the pipeline where
alleles are tallied, and it feeds QC and the allelic spectrum, **not** the test.

The three non-zero totals become three ALTs on one multi-allelic record. `RU_DOSAGE`
carries each ALT's dosage in repeat units at round-trip precision; the genotype indices
say which total each haplotype holds.

In [ ]:
loci_vcf = (WORK / "loci.vcf").read_text()
rec = [ln.split("\t") for ln in loci_vcf.splitlines() if not ln.startswith("#")][0]
for name, val in zip(["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT"], rec[:9]):
    print(f"{name:7s} {val}")
print()
for s, cell in zip(SAMPLES, rec[9:]):
    print(f"{s:7s} {cell}")

## Level 3 — per ancestry, SUM the dosages and COUNT the haplotypes

This is the step most easily mis-stated. `site_dosages()` keeps **two** accumulators:

```python
x = units[allele]     # the signed dosage in repeat units
ds[anc]  += x         # DS{k}: SUM of signed dosages on ancestry-k haplotypes
cnt[anc] += 1         # ANC{k}: COUNT of ancestry-k haplotypes
```

`DS{k}` is continuous and can be negative or zero; `ANC{k}` is a count of haplotypes
regardless of what they carry. The allele's identity is gone the moment `units[allele]`
is looked up — there is no per-allele bookkeeping past this point.

In [ ]:
info = wad.info_dict(rec[7])
units_ru = wad.allele_units(info, rec[3], rec[4])
print("INFO RU_DOSAGE ->", info["RU_DOSAGE"])
print("units by allele index (0 = REF):", units_ru)
print("is_ru_test:", wad.is_ru_test(info))

ds_rows, anc_rows, max_abs, n_missing, n_hap = wad.site_dosages(
    sample_fields=rec[9:], fmt_keys=rec[8].split(":"), units=units_ru,
    num_ancs=3, where="L1", missing="ref",
)
print(f"\nRU_SCALE = max |x| = {max_abs}   n_hap = {n_hap}   n_missing = {n_missing}")

ANC_NAME = {0: "AFR", 1: "EUR", 2: "AMR"}
level3 = pd.DataFrame(
    {**{f"DS_{ANC_NAME[k]}": [r[k] / max_abs for r in ds_rows] for k in range(3)},
     **{f"ANC_{ANC_NAME[k]}": [r[k] for r in anc_rows] for k in range(3)}},
    index=SAMPLES)
level3

`ds_rows` as returned is the **unscaled** sum; the writer divides by `RU_SCALE` on output,
which is what the table above shows. Score tests are invariant to that scale — multiply
FELIX `BETA` by `1/RU_SCALE` to get an effect per repeat unit.

### The VCF FELIX step 2 actually reads

In [ ]:
buf = io.StringIO()
stats = wad.write_admixed_vcf(str(WORK / "loci.vcf"), buf, num_ancs=3)
admixed = buf.getvalue()
(WORK / "ru.admixed.vcf").write_text(admixed)

print(json.dumps(stats, indent=2))
print()
print("\n".join(ln for ln in admixed.splitlines() if ln.startswith("#CHROM") or not ln.startswith("#")))

In [ ]:
arec = [ln.split("\t") for ln in admixed.splitlines() if not ln.startswith("#")][0]
keys = arec[8].split(":")
fields = pd.DataFrame([dict(zip(keys, c.split(":"))) for c in arec[9:]], index=SAMPLES)
print("INFO:", arec[7])
fields

## The cancellation case

S4 carries +6 bp and −6 bp on two haplotypes that FLARE assigns to the **same** ancestry.
They sum to zero, so S4's `DS`/`ANC` row is byte-identical to S5's — a sample with two
reference AFR haplotypes. A double carrier and a non-carrier are the same observation
under this coding.

In [ ]:
pair = fields.loc[["S4", "S5"]]
print(pair.to_string())
assert pair.loc["S4"].tolist() == pair.loc["S5"].tolist(), "expected S4 to be indistinguishable from S5"
print("\nS4 == S5 under the ru coding:", True)

print("\nBut S2's +6 bp (AFR) and -3 bp (EUR) land in different bins and cannot cancel:")
print(fields.loc[["S2"]].to_string())

This is the mechanism behind the flat dosage-vs-carrier contrast in the interim scan: where
expansions and contractions balance within an ancestry, the dosage and carrier codings are
not stronger and weaker versions of one test — they are *orthogonal*, asking different
questions.

It also shows why the coding is never shifted. A per-haplotype shift `c` adds `c · ANC{k}`
to `DS{k}`, so the per-ancestry tests would stop measuring repeat length and start
measuring local ancestry.

## The biallelic comparator arm

`coding="carrier"` is the comparator used by the `ru_biallelic` arm. It changes exactly one
thing: every allele with a non-zero repeat dosage is mapped to `x = 1`. Same loci, same
carrier haplotypes, same ancestry assignment, same missingness handling — only the weighting
by repeat count differs.

In [ ]:
print("ru     :", wad.apply_coding(units_ru, "ru"))
print("carrier:", wad.apply_coding(units_ru, "carrier"))

cbuf = io.StringIO()
wad.write_admixed_vcf(str(WORK / "loci.vcf"), cbuf, num_ancs=3, coding="carrier")
crec = [ln.split("\t") for ln in cbuf.getvalue().splitlines() if not ln.startswith("#")][0]
ckeys = crec[8].split(":")
carrier = pd.DataFrame([dict(zip(ckeys, c.split(":"))) for c in crec[9:]], index=SAMPLES)

side = pd.concat(
    {"ru": fields[[f"DS{k}" for k in (1, 2, 3)]],
     "carrier": carrier[[f"DS{k}" for k in (1, 2, 3)]]}, axis=1)
print("\ncarrier INFO:", crec[7])
side

Under `carrier`, `RU_SCALE` becomes 1 and `DS{k}` is the ancestry-k count of non-reference
haplotypes (0–2). Two differences show up in the table above:

- **The sign is discarded.** S2 and S3 each carry a −3 bp contraction on a EUR haplotype:
  `DS2 = -0.5` under `ru`, `DS2 = +1` under `carrier`. A contraction and an expansion
  become the same observation.
- **Cancellation disappears.** S4 gives `DS1 = 0` under `ru` and `DS1 = 2` under `carrier`,
  because two carrier haplotypes cannot cancel when both count as +1.

The carrier *haplotype* set is identical between the arms by construction — the same
haplotypes are non-reference in both — but the sample-level `DS{k}` is not, exactly at
cancelling samples like S4.

> **Unverified:** whether the `min_carriers = 20` gate (`--dosage-qc carrier`) keys off the
> carrier haplotype set or off non-zero sample-level `DS{k}` decides whether the two arms can
> end up testing slightly different locus sets. `FELIX_DOSAGE_QC` is not implemented in the
> `repeat-sv-dosage` checkout at `ee95abe` — it lives in the container image — so this
> notebook cannot settle it. Worth confirming against `felix-pilot:0.2.0` directly.

## What step 2 does with it

FELIX reads `FORMAT DS1..DSK` as the dosage matrix with `ANC1..ANCK` alongside:

```
--vcfField=DS  --is_admixed=TRUE  --number_of_ancestry=K
```

and emits one result row per locus containing

- `K` per-ancestry score tests on `DS{k}`
- `P_hom_admixed` — the pooled 1-df test on `Σₖ DS{k}`
- `P_het_admixed` — effects allowed to differ across ancestries
- `P_cct_admixed` — the Cauchy combination of `P_het_admixed` and `P_hom_admixed`

A locus with 12 distinct haplotype totals still yields one test per ancestry, because the
totals enter as numbers on a line rather than as 12 categories. That is what makes this a
dosage method — and why it buys nothing at a locus where all carriers share one length,
and asks a different question where gains and losses cancel.

In [ ]:
print("files written:")
for p in sorted(WORK.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size:>8,d} bytes")
print()
print(json.dumps(json.loads((WORK / "loci.summary.json").read_text())["counts"], indent=2))